# exp_6 · 03 decision — measured and stress settings, H1, VOI grid, touch context
H1 = G1 & G2 & G3 & (measured paired 95 % CI low > 0 on provider **and** `L_ctrl`) & (stress mean Δ ≥ −0.005 on
both). Saved exp_2 weights, exp_3's per-fold `target_like` decoder selections; no fit, no re-selection.
Reads only `results/*.json` and `cache/run_lengths.npy` from 02 (aggregate run lengths).

In [1]:
from pathlib import Path
import os, sys, json, time, subprocess
from datetime import datetime, timezone
ROOT = Path.cwd()
if not (ROOT / 'exp6_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_6')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import exp6_common as C
import exp1_common as C1
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
def now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')
def make_log(name):
    path = C.LOG_DIR / f'{name}_progress.log'
    path.parent.mkdir(parents=True, exist_ok=True)
    def log(message, **_):
        print(message, flush=True)
        with path.open('a') as handle:
            handle.write(f'{now()} {message}\n')
    return log
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])

GUARD_FILE = 'guard_03.json'
guard = C.guard_check(C.RESULTS / 'write_guard_before.json', C.RESULTS / GUARD_FILE)
print('nothing written under exp_0-exp_5 / data since freezing:', guard['all_unchanged'],
      {k: v['unchanged'] for k, v in guard['folders'].items()})
if not guard['all_unchanged']:
    raise RuntimeError('Write guard changed: diagnose before continuing')

import exp6_decision as DE
import exp6_sim as S
log = make_log('03')
quality = C1.read_json(C.RESULTS / 'detector_quality.json')
q = {'coverage': quality['coverage'], 'sigma_m': quality['test']['sigma_m'], 'fp_rate': quality['test']['fp_rate']}
lengths = np.load(C.CACHE / 'run_lengths.npy')
draw = lengths if len(lengths) else np.array([1])
print('measured:', S.measured(q), '| stress:', S.stress(q), '| accepted runs:', len(lengths))

PREREG verified dca69fd7188c7ef8 | frozen 2026-11-10T14:57:58+00:00


nothing written under exp_0-exp_5 / data since freezing: True {'exp_0': True, 'exp_1': True, 'exp_2': True, 'exp_3': True, 'exp_4': True, 'exp_5': True, 'data': True, 'data_metric_repo': True}
measured: Setting(name='measured', coverage=0.31160373126247237, noise=0.48320129491003044, fp=0.23963775687913624) | stress: Setting(name='stress', coverage=0.15580186563123619, noise=0.9664025898200609, fp=0.5292755137582725) | accepted runs: 671


In [2]:
settings = [S.BASE, S.measured(q), S.stress(q)]
rows = S.evaluate(settings, draw, 'reference', log)
measured = {label: S.deltas(rows, 'measured', label) for label in C.LABEL_SETS}
stressed = {label: S.deltas(rows, 'stress', label) for label in C.LABEL_SETS}
g1 = C1.read_json(C.RESULTS / 'g1.json')['share']
gates = DE.gate_table(g1, quality['test']['precision'], quality['test']['recall'], quality['coverage'])
decision = DE.decide(gates, DE.ci_clause(measured), DE.stress_clause(stressed))
decision.update({'decided_at_utc': now(), 'tau': quality['tau'],
                 'settings': {'measured': S.measured(q).as_dict(), 'stress': S.stress(q).as_dict()},
                 'files_written': False,
                 'candidate_files': 'written by 04_target' if decision['h1_passed'] else 'none: H1 failed'})
C.write_results(C.RESULTS / 'decision.json', decision)
print('gates:', {k: v['pass'] for k, v in gates.items()})
for label in C.LABEL_SETS:
    print(label, 'measured', {k: decision['ci_clause'][label][k] for k in ('mean_delta', 'ci_low', 'ci_high', 'pass')},
          '| stress', {k: decision['stress_clause'][label][k] for k in ('mean_delta', 'pass')})
print('H1 passed:', decision['h1_passed'], '| failed:', decision['failed'])

reference target_like J03WMX 3.4s


reference measured J03WMX 2.8s


reference stress J03WMX 2.3s


reference target_like J03WN1 2.0s


reference measured J03WN1 2.3s


reference stress J03WN1 1.9s


... (12 outputs left out)

reference target_like J03WR9 1.9s


reference measured J03WR9 2.6s


reference stress J03WR9 2.1s


gates: {'G1': True, 'G2': False, 'G3': True}
provider measured {'mean_delta': -0.12456325839837216, 'ci_low': -0.13797952871197022, 'ci_high': -0.10754503802231265, 'pass': False} | stress {'mean_delta': -0.10588500336029297, 'pass': False}
ctrl measured {'mean_delta': -0.14507329776549074, 'ci_low': -0.16192020655656703, 'ci_high': -0.12591764578442102, 'pass': False} | stress {'mean_delta': -0.10920149993603175, 'pass': False}
H1 passed: False | failed: ['G2', 'ci_clause:provider', 'ci_clause:ctrl', 'stress_clause:provider', 'stress_clause:ctrl']


In [3]:
voi = S.voi_grid(q) if len(lengths) else []
voi_rows = S.evaluate([S.BASE] + voi, draw, 'reference', log) if voi else []
grid = [{**s.as_dict(), **{label: DE.delta_summary(S.deltas(voi_rows, s.name, label)) for label in C.LABEL_SETS}}
        for s in voi]
C.write_results(C.RESULTS / 'voi_grid.json', {'computed_at_utc': now(), 'report_only': True,
                                            'computable': bool(voi), 'grid': grid})
for cell in grid:
    print(cell['name'], {label: round(cell[label]['mean_delta'], 4) for label in C.LABEL_SETS})

reference target_like J03WMX 1.7s


reference voi_c0.12_s0_f0 J03WMX 1.8s


reference voi_c0.12_s0_fhat J03WMX 1.7s


reference voi_c0.12_shat_f0 J03WMX 1.8s


reference voi_c0.12_shat_fhat J03WMX 1.9s


reference voi_c0.25_s0_f0 J03WMX 2.2s


... (110 outputs left out)

reference voi_c1_s0_fhat J03WR9 9.1s


reference voi_c1_shat_f0 J03WR9 8.9s


reference voi_c1_shat_fhat J03WR9 9.0s


voi_c0.12_s0_f0 {'provider': 0.0072, 'ctrl': 0.0108}
voi_c0.12_s0_fhat {'provider': -0.0223, 'ctrl': -0.0221}
voi_c0.12_shat_f0 {'provider': -0.0368, 'ctrl': -0.0414}
voi_c0.12_shat_fhat {'provider': -0.0538, 'ctrl': -0.0596}
voi_c0.25_s0_f0 {'provider': 0.0183, 'ctrl': 0.0186}
voi_c0.25_s0_fhat {'provider': -0.0431, 'ctrl': -0.0422}
voi_c0.25_shat_f0 {'provider': -0.0692, 'ctrl': -0.0905}
voi_c0.25_shat_fhat {'provider': -0.0978, 'ctrl': -0.1138}
voi_c0.5_s0_f0 {'provider': 0.0349, 'ctrl': 0.0334}
voi_c0.5_s0_fhat {'provider': -0.092, 'ctrl': -0.0915}
voi_c0.5_shat_f0 {'provider': -0.1257, 'ctrl': -0.1606}
voi_c0.5_shat_fhat {'provider': -0.1872, 'ctrl': -0.2069}
voi_c1_s0_f0 {'provider': 0.0788, 'ctrl': 0.0562}
voi_c1_s0_fhat {'provider': -0.1767, 'ctrl': -0.175}
voi_c1_shat_f0 {'provider': -0.1622, 'ctrl': -0.2055}
voi_c1_shat_fhat {'provider': -0.2989, 'ctrl': -0.2857}


In [4]:
touch_rows = S.evaluate([S.BASE, S.measured(q)], draw, 'touch', log)
touch = {'report_only': True, 'recheck_target_like': S.recheck_base(touch_rows, 'touch'),
         'measured': {label: DE.delta_summary(S.deltas(touch_rows, 'measured', label, 'touch')) for label in C.LABEL_SETS}}
C.write_results(C.RESULTS / 'touch_context.json', {'computed_at_utc': now(), **touch})
pd.DataFrame(rows + voi_rows + touch_rows).to_csv(C.RESULTS / 'sim_rows_03.csv', index=False)
print('touch context:', {label: round(touch['measured'][label]['mean_delta'], 4) for label in C.LABEL_SETS},
      touch['recheck_target_like'])

touch target_like J03WMX 1.8s


touch measured J03WMX 3.4s


touch target_like J03WN1 1.8s


touch measured J03WN1 2.4s


touch target_like J03WOH 1.7s


touch measured J03WOH 2.2s


... (5 outputs left out)

touch measured J03WQQ 2.8s


touch target_like J03WR9 2.0s


touch measured J03WR9 2.5s


touch context: {'provider': -0.1514, 'ctrl': -0.1991} {'rows': 14, 'max_abs_diff': 0.0, 'rows_equal': True}
